In [2]:
!pip install groq
from groq import Groq
client = Groq(
    api_key="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"
)
python_code = """
def student_grade_report(students):
    total_marks = 0
    for student in students:
        marks = sum(student["marks"])
        average = marks / len(student["marks"])
        if average >= 90:
            grade = "A"
        elif average >= 75:
            grade = "B"
        elif average >= 50:
            grade = "C"
        else:
            grade = "Fail"
        print(f"Name: {student['name']}")
        print(f"Total Marks: {marks}")
        print(f"Average: {average}")
        print(f"Grade: {grade}")
        print("-" * 30)
        total_marks += average
    class_average = total_marks / len(students)
    print(f"Class Average: {class_average}")
"""
zero_shot_prompt = f"""
Review the following Python code and suggest improvements:
{python_code}
"""
few_shot_prompt = f"""
Example:
Code:
def add(a,b):
 return a+b
Review:
- Add spaces after commas for readability.
- Add a function docstring.
- Use meaningful formatting.
Now review this code:
{python_code}
"""
cot_prompt = f"""
Review the following Python code step-by-step.
1. First identify possible bugs.
2. Then check code style and readability.
3. Then check security or robustness issues.
4. Finally provide overall improvements.
Code:
{python_code}
"""
def get_review(prompt):
    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[
            {
                "role": "user",
                "content": prompt
            }
        ],
        temperature=0.5,
        max_tokens=500
    )
    return response.choices[0].message.content
zero_shot_output = get_review(zero_shot_prompt)
few_shot_output = get_review(few_shot_prompt)
cot_output = get_review(cot_prompt)
print("=" * 70)
print("ZERO-SHOT OUTPUT")
print("=" * 70)
print(zero_shot_output)
print("\n\n")
print("=" * 70)
print("FEW-SHOT OUTPUT")
print("=" * 70)
print(few_shot_output)
print("\n\n")
print("=" * 70)
print("CHAIN-OF-THOUGHT OUTPUT")
print("=" * 70)
print(cot_output)

ZERO-SHOT OUTPUT
### Code Review and Improvements

The provided Python code generates a student grade report based on their marks. Here's a reviewed and improved version of the code:

#### Issues with the Original Code:

1.  **Error Handling**: The code does not handle potential errors, such as division by zero or missing keys in the student dictionary.
2.  **Magic Numbers**: The code uses magic numbers (e.g., 90, 75, 50) that are not self-explanatory. Defining these numbers as constants would improve readability.
3.  **Code Organization**: The code mixes calculation and printing logic, making it harder to maintain and test.

#### Improved Code:

```python
def calculate_average(marks):
    """Calculate the average of a list of marks."""
    if not marks:
        raise ValueError("Cannot calculate average of empty list")
    return sum(marks) / len(marks)

def determine_grade(average):
    """Determine the grade based on the average."""
    GRADE_THRESHOLDS = {
        "A": 90,
        